In [8]:
import pandas as pd
import numpy as np
from collections import Counter
import ast
import matplotlib.pyplot as plt
import torch
import math
import seaborn as sns
from pathlib import Path
import os

### 1. Folder and file definitions
Based on the threshold analysis in threshold_analysis.ipynb, we choose the threshold of ... % and use this throughout the entire analysis in this file.

In [2]:
RESULT_DIR = Path("../results/oct24_GCN_persuade_wan_cooc_qwen2515b_wan-w5_nf-emb_ew-wan_sp-random_L2_H64_sum_lr0.001_seed2029")
motif_file = RESULT_DIR / "noob_motcom040.csv" 
results_file = RESULT_DIR / "explanations_none_object.pth"

In [4]:
df_motifs = pd.read_csv(motif_file)
df_motifs

,motif,human_present,human_selected,human_rate,machine_present,machine_selected,machine_rate,difference
0,"((0, 1), (0, 164))",1,0,0.0,1,1,1.0,1.0
1,"((273, 179), (273, 272))",1,1,1.0,1,0,0.0,-1.0
2,"((0, 1), (0, 265))",3,0,0.0,1,1,1.0,1.0
3,"((44, 198), (113, 44))",1,0,0.0,1,1,1.0,1.0
4,"((45, 168), (108, 168))",1,1,1.0,1,0,0.0,-1.0
...,...,...,...,...,...,...,...,...
273668,"((275, 113), (275, 217))",2,1,0.5,0,0,NaN,NaN
273669,"((275, 154), (275, 268))",1,1,1.0,0,0,NaN,NaN
273670,"((275, 163), (275, 217))",2,2,1.0,0,0,NaN,NaN
273671,"((275, 163), (275, 224))",1,1,1.0,0,0,NaN,NaN


### 2. Summary motifs

In [5]:
human_selected = df_motifs["human_selected"] > 0
machine_selected = df_motifs["machine_selected"] > 0

human_only = (human_selected & ~machine_selected).sum()
machine_only = (~human_selected & machine_selected).sum()
shared = (human_selected & machine_selected).sum()

total = (human_selected | machine_selected).sum()

print(f"Total unique selected motifs: {total}")
print(f"Human only: {human_only}")
print(f"Machine only: {machine_only}")
print(f"Shared: {shared}")

Total unique selected motifs: 273673
Human only: 221552
Machine only: 33872
Shared: 18249


In [6]:
h = df_motifs[
    (df_motifs["human_selected"] > 0) &
    (df_motifs["machine_selected"] == 0)
]

m = df_motifs[
    (df_motifs["human_selected"] == 0) &
    (df_motifs["machine_selected"] > 0)
]

print("Human-only motifs selected once (uniqe motifs within the human class):",
      (h["human_selected"] == 1).sum())

print("Amount of non-unique motifs within the human class: ",
      human_only - (h["human_selected"] == 1).sum())

print("Machine-only motifs selected once (unique motifs within the machine class):",
      (m["machine_selected"] == 1).sum())

print("Amount of non-unique motifs within the machine class: ",
      machine_only - (m["machine_selected"] == 1).sum())

Human-only motifs selected once (uniqe motifs within the human class): 170894
Amount of non-unique motifs within the human class:  50658
Machine-only motifs selected once (unique motifs within the machine class): 28406
Amount of non-unique motifs within the machine class:  5466


In [9]:
results = torch.load(
    results_file,
    map_location="cpu",
    weights_only=False
)

In [10]:
results

[{'row': 13,
  'label': 0,
  'pred': 0,
  'prob_machine': 0.1268298,
  'x': tensor([[-0.2709,  0.0440, -0.0203,  ..., -0.4923,  0.6369,  0.2364],
          [ 0.6604,  0.6389,  0.8626,  ..., -0.5590,  1.0752,  0.5770],
          [-0.1310,  0.5548,  0.5304,  ..., -0.4327,  0.6935, -0.1450],
          ...,
          [-0.2059,  0.8924,  0.4946,  ..., -0.0663, -0.2220, -0.1561],
          [-0.5936, -0.0749,  0.5304,  ...,  0.2310,  0.5684,  0.4886],
          [-1.1428,  0.1462,  0.1303,  ...,  0.6045,  0.3083,  0.9695]]),
  'edge_index': tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   8,   8,   8,   8,
             8,   8,   8,   9,   9,   9,   9,   9,   9,   9,  10,  10,  10,  10,
            10,  10,  12,  12,  12,  12,  12,  13,  13,  13,  13,  13,  13,  13,
            16,  16,  16,  20,  20,  20,  20,  20,  21,  21,  21,  21,  21,  21,
            21,  21,  21,  21,  21,  21,  21,  21,  21,  21,  21,  21,  21,  21,
            21,  21,  21,  29,  29,  29,  29,  29,  29,  2

In [12]:
df_explanations = pd.DataFrame([
    {
        "row": result["row"],
        "label": result["label"],
        "pred": result["pred"]
    }
    for result in results
])

# Count unique graphs per class
graph_counts = (
    df_explanations
    .groupby("label")["row"]
    .nunique()
)

print("Number of graphs per class:")
print(graph_counts)

print("Total unique graphs:", df_explanations["row"].nunique())

Number of graphs per class:
label
0    180
1    180
Name: row, dtype: int64
Total unique graphs: 360


In [13]:
df_correct = df_explanations[
    df_explanations["label"] == df_explanations["pred"]
]

n_human_graphs = df_correct[
    df_correct["label"] == 0
]["row"].nunique()

n_machine_graphs = df_correct[
    df_correct["label"] == 1
]["row"].nunique()

print("Correctly classified human graphs:", n_human_graphs)
print("Correctly classified machine graphs:", n_machine_graphs)

Correctly classified human graphs: 176
Correctly classified machine graphs: 178
